# 01 — Bases de l'API Riot Games

Objectif de ce notebook : apprendre à interroger l'API Riot (HTTP, authentification, rate limiting) et en extraire une composition d'équipe. On avance concept par concept.

**Partie 1 : pourquoi l'authentification est nécessaire.**

## Concept 1 — Anatomie d'une requête HTTP

Une requête HTTP, c'est : une **méthode** (ici `GET`, on demande une info sans rien modifier), une **URL** (l'adresse de la ressource), et des **headers** (métadonnées, ex: qui je suis, quel format je veux).

On appelle l'endpoint "platform status" de Riot — il indique si les serveurs d'une région ont des problèmes. On choisit cet endpoint car il ne dépend d'aucun pseudo : juste une plateforme (`euw1`).

**On l'appelle volontairement SANS clé API pour l'instant**, pour voir ce que Riot répond.

In [ ]:
import requests

platform = "euw1"
url = f"https://{platform}.api.riotgames.com/lol/status/v4/platform-data"

response = requests.get(url)

print("Status code:", response.status_code)
print("Corps de la réponse:", response.text)

**Ce que tu devrais observer** : un `status_code` de `401` (Unauthorized) ou `403` (Forbidden), avec un message d'erreur du type `"Unauthorized"`. Le serveur refuse de répondre tant qu'on ne prouve pas qui on est — normal, l'API Riot est protégée pour éviter les abus et attribuer l'usage à chaque application.

C'est le rôle de l'**authentification**, qu'on ajoute au prochain concept.

## Concept 2 — Authentification via header

Riot attend la clé API dans un **header** nommé `X-Riot-Token` (pas dans l'URL). Pourquoi un header et pas l'URL ?
- Les URLs se retrouvent souvent loguées en clair (logs serveur, historique de navigateur, proxys) — un secret dans l'URL fuite facilement.
- Les headers sont faits pour transporter des métadonnées comme l'authentification, séparées de la ressource demandée.

On charge la clé depuis `.env` avec `python-dotenv`, plutôt que de l'écrire en dur dans le notebook — ainsi le notebook reste partageable/versionnable sans exposer ton secret.

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()  # lit le fichier .env à la racine du projet et charge ses variables

api_key = os.environ["RIOT_API_KEY"]

headers = {"X-Riot-Token": api_key}

response = requests.get(url, headers=headers)

print("Status code:", response.status_code)
print("Corps de la réponse:", response.json())

**Ce que tu devrais observer** : un `status_code` de `200` (OK) cette fois, avec un JSON contenant l'état de la plateforme EUW (généralement une liste vide d'incidents si tout va bien).

## Concept 3 — Régions vs plateformes

Riot utilise **deux systèmes de routage différents**, source de confusion classique :
- Les **plateformes** (`euw1`, `na1`, `kr`...) : utilisées par des endpoints comme le statut serveur (ci-dessus), les ligues, l'historique d'invocateur.
- Les **régions** (`europe`, `americas`, `asia`, `sea`) : utilisées par les endpoints plus récents comme Account-V1 et Match-V5. Chaque région regroupe plusieurs plateformes (`europe` couvre `euw1`, `eun1`, `tr1`, `ru`).

Ta plateforme `euw1` correspond donc à la région `europe` pour les appels qui suivent.

## Concept 4 — Account-V1 : résoudre un Riot ID en `puuid`

Le `puuid` est l'identifiant unique et stable d'un joueur chez Riot — c'est ce que les autres endpoints (comme Match-V5) attendent, pas le pseudo directement. Account-V1 est le seul moyen de passer d'un Riot ID (`pseudo#tag`) à un `puuid`.

On encode le pseudo et le tag avec `urllib.parse.quote` : un Riot ID peut contenir des espaces ou caractères spéciaux, invalides tels quels dans une URL.

In [ ]:
from urllib.parse import quote

region = "europe"  # routage régional, différent de la plateforme "euw1"
game_name = "Spear Shot"
tag_line = "1111"

account_url = (
    f"https://{region}.api.riotgames.com/riot/account/v1/accounts/by-riot-id/"
    f"{quote(game_name)}/{quote(tag_line)}"
)

response = requests.get(account_url, headers=headers)
print("Status code:", response.status_code)

account_data = response.json()
print(account_data)

puuid = account_data["puuid"]
print("PUUID:", puuid)

**Ce que tu devrais observer** : `status_code` `200`, un JSON avec ton `puuid`, `gameName` et `tagLine`.

---

**Checkpoint** : exécute ces deux nouvelles cellules et confirme que tu obtiens bien ton `puuid`. Ensuite on passe à Match-V5 : lister tes matchs récents et récupérer une vraie composition d'équipe.

## Concept 5 — Match-V5 : lister puis détailler des matchs

Récupérer des matchs se fait en **deux appels** :
1. `GET /lol/match/v5/matches/by-puuid/{puuid}/ids` → une liste d'IDs de matchs récents (ex: `"EUW1_1234567890"`)
2. `GET /lol/match/v5/matches/{matchId}` → le détail complet d'**un** match

Pourquoi deux appels et pas un seul qui renvoie tout directement ? Parce que le détail d'un match est un gros JSON (stats complètes de 10 joueurs). Si l'endpoint de liste renvoyait tout ce détail pour chaque match, chaque appel serait énorme même quand on ne veut que 5 IDs. Séparer les deux permet de ne payer le "coût" du détail que pour les matchs qui nous intéressent vraiment.

On utilise le même routage régional (`europe`) que pour Account-V1.

In [ ]:
match_ids_url = f"https://{region}.api.riotgames.com/lol/match/v5/matches/by-puuid/{puuid}/ids"

response = requests.get(match_ids_url, headers=headers, params={"start": 0, "count": 5})
print("Status code:", response.status_code)

match_ids = response.json()
print(match_ids)

**Ce que tu devrais observer** : `status_code` `200` et une liste de 5 chaînes comme `"EUW1_..."`.

Remarque sur `params={"start": 0, "count": 5}` : c'est différent de `quote(game_name)` utilisé plus haut. Ici, `start` et `count` sont des **paramètres de requête** (query parameters) — des options qui suivent un `?` dans l'URL (ex: `...ids?start=0&count=5`), pas une partie de l'identité de la ressource comme le `puuid`. `requests` les construit et les encode automatiquement à partir du dict `params`, donc pas besoin de `quote` ici.

In [ ]:
first_match_id = match_ids[0]
match_detail_url = f"https://{region}.api.riotgames.com/lol/match/v5/matches/{first_match_id}"

response = requests.get(match_detail_url, headers=headers)
print("Status code:", response.status_code)

match_data = response.json()
print("Clés de premier niveau:", match_data.keys())
print("Clés dans 'info':", match_data["info"].keys())
print("Nombre de participants:", len(match_data["info"]["participants"]))

**Ce que tu devrais observer** : `status_code` `200`, et `info.participants` avec **10** entrées (les 10 joueurs de la partie, 5 par équipe).

La structure d'un match Riot :
- `metadata` : infos techniques (ID du match, version des données)
- `info.participants` : liste de 10 dicts, un par joueur — chacun contient `championName`, `teamId` (100 = bleue, 200 = rouge), `win` (booléen), plus des dizaines d'autres stats (dégâts, CS, items...)

---

**Checkpoint** : exécute ces trois nouvelles cellules et confirme que tu vois bien 10 participants. Ensuite : rate limiting (pourquoi/quand un `429` arrive) puis extraction propre de la composition (juste champion + équipe + résultat, sans le bruit des autres stats).

## Concept 6 — Rate limiting

Riot limite le nombre d'appels par clé (ex: pour une clé de dev, 20 requêtes/seconde et 100 requêtes/2 minutes) — ça protège leur infrastructure et répartit l'usage équitablement entre toutes les applications qui utilisent l'API.

Chaque réponse contient des headers qui indiquent où tu en es (`X-App-Rate-Limit-Count`). Si tu dépasses la limite, Riot répond avec `status_code = 429` et un header `Retry-After` : le nombre de secondes à attendre avant de réessayer.

On regarde d'abord ces headers sur une réponse qu'on a déjà, puis on écrit une petite fonction qui gère automatiquement le `429` en attendant et en réessayant.

In [ ]:
for key, value in response.headers.items():
    if "rate-limit" in key.lower():
        print(key, ":", value)

In [ ]:
import time


def riot_get(url, params=None, max_retries=3):
    for attempt in range(max_retries):
        response = requests.get(url, headers=headers, params=params)
        if response.status_code == 429:
            retry_after = int(response.headers.get("Retry-After", 1))
            print(f"Rate limit atteint, attente de {retry_after}s avant de réessayer...")
            time.sleep(retry_after)
            continue
        return response
    raise RuntimeError("Trop de tentatives après rate limiting répété")


# on réutilise cette fonction à la place de requests.get direct, à partir de maintenant
response = riot_get(match_detail_url)
print("Status code:", response.status_code)

## Concept 7 — Extraire une composition propre

`match_data["info"]["participants"]` contient énormément de stats par joueur (dégâts, or, items, wards...). Pour l'instant, on ne veut que ce qui définit une **composition** : le champion, l'équipe, le rôle et le résultat. On isole ça dans une structure Python simple (liste de dicts), sans encore utiliser pandas — ça sera l'objet de la phase 2.

`teamId` vaut `100` pour l'équipe bleue et `200` pour l'équipe rouge (convention Riot). `teamPosition` donne le rôle (`TOP`, `JUNGLE`, `MIDDLE`, `BOTTOM`, `UTILITY` pour support) — utile à garder dès maintenant, on en aura besoin plus tard pour le modèle.

In [ ]:
def extract_composition(match_data):
    composition = []
    for p in match_data["info"]["participants"]:
        composition.append({
            "champion": p["championName"],
            "team": "blue" if p["teamId"] == 100 else "red",
            "role": p["teamPosition"],
            "win": p["win"],
        })
    return composition


composition = extract_composition(match_data)
for row in composition:
    print(row)

## Concept 8 — Mise en perspective

Ce qu'on vient de faire (`Riot ID → puuid → liste de matchs → détail → composition`) donne **une seule** ligne de données d'entraînement : une composition + son résultat.

Pour la phase suivante, il faudra répéter ça sur des **centaines/milliers de matchs**, de **joueurs différents** (pas juste toi) — c'est ce qui construira le dataset qui alimentera la Factorization Machine (voir la réflexion ML qu'on a eue : les embeddings de champions ont besoin de voir beaucoup de paires de champions ensemble pour apprendre des synergies fiables).

Important à garder en tête : cette collecte via un pseudo n'a rien à voir avec le produit final — l'utilisateur de l'assistant ne donnera jamais un pseudo, juste une composition hypothétique. Le pseudo n'a servi que de point d'entrée technique pour apprendre l'API et amorcer la collecte de données.

---

## Phase 1 terminée ✅

Tu sais maintenant : faire une requête HTTP authentifiée, gérer le rate limiting, naviguer entre routage plateforme/région, et extraire une composition d'un match réel.

**Prochaine étape (phase 2)** : construire le pipeline de collecte à grande échelle (plusieurs joueurs, plusieurs matchs) et structurer tout ça avec pandas en un vrai dataset.